## Grouped Aggregations on a Spark Data Frame

In [2]:
%run './01 Validate Data Sets for Aggregations using Spark.ipynb'

24/03/24 16:52:44 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/03/24 16:52:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/03/24 16:52:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/03/24 16:52:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/03/24 16:52:46 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/03/24 16:52:46 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
24/03/24 16:52:46 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


root
 |-- order_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)

root
 |-- order_item_id: integer (nullable = true)
 |-- order_item_order_id: integer (nullable = true)
 |-- order_item_product_id: integer (nullable = true)
 |-- order_item_quantity: short (nullable = true)
 |-- order_item_subtotal: float (nullable = true)
 |-- order_item_product_price: float (nullable = true)



In [3]:
order_items_grouped = order_items.groupBy('order_item_order_id')

In [4]:
type(order_items_grouped)

pyspark.sql.group.GroupedData

In [5]:
order_items_grouped.count().show()

+-------------------+-----+
|order_item_order_id|count|
+-------------------+-----+
|                148|    3|
|                463|    4|
|                471|    2|
|                496|    5|
|               1088|    2|
|               1580|    1|
|               1591|    3|
|               1645|    5|
|               2366|    1|
|               2659|    5|
|               2866|    4|
|               3175|    2|
|               3749|    1|
|               3794|    1|
|               3918|    4|
|               3997|    2|
|               4101|    1|
|               4519|    1|
|               4818|    1|
|               4900|    2|
+-------------------+-----+
only showing top 20 rows



In [6]:
order_items_grouped.count().withColumnRenamed('count', 'orders_count').show()

+-------------------+------------+
|order_item_order_id|orders_count|
+-------------------+------------+
|                148|           3|
|                463|           4|
|                471|           2|
|                496|           5|
|               1088|           2|
|               1580|           1|
|               1591|           3|
|               1645|           5|
|               2366|           1|
|               2659|           5|
|               2866|           4|
|               3175|           2|
|               3749|           1|
|               3794|           1|
|               3918|           4|
|               3997|           2|
|               4101|           1|
|               4519|           1|
|               4818|           1|
|               4900|           2|
+-------------------+------------+
only showing top 20 rows



In [7]:
order_items_grouped.sum().show()

+-------------------+------------------+------------------------+--------------------------+------------------------+------------------------+-----------------------------+
|order_item_order_id|sum(order_item_id)|sum(order_item_order_id)|sum(order_item_product_id)|sum(order_item_quantity)|sum(order_item_subtotal)|sum(order_item_product_price)|
+-------------------+------------------+------------------------+--------------------------+------------------------+------------------------+-----------------------------+
|                148|              1047|                     444|                      1407|                       8|      479.99000549316406|           229.99000549316406|
|                463|              4522|                    1852|                      1685|                      13|       829.9200096130371|           249.97000122070312|
|                471|              2307|                     942|                      1030|                       2|      169.98000717

In [9]:
# only shows numeric datatype sum ignoring other data types
orders \
    .groupBy('order_id') \
    .sum() \
    .show()

+--------+-------------+----------------------+
|order_id|sum(order_id)|sum(order_customer_id)|
+--------+-------------+----------------------+
|     148|          148|                  5383|
|     463|          463|                  8709|
|     471|          471|                 10861|
|     496|          496|                   615|
|     833|          833|                  4076|
|    1088|         1088|                 11139|
|    1238|         1238|                  2610|
|    1342|         1342|                  8350|
|    1580|         1580|                  9713|
|    1591|         1591|                 10666|
|    1645|         1645|                  4709|
|    1829|         1829|                  3951|
|    1959|         1959|                 10066|
|    2122|         2122|                 10518|
|    2142|         2142|                  1594|
|    2366|         2366|                  7887|
|    2659|         2659|                  6630|
|    2866|         2866|                

In [12]:
order_items_grouped = order_items \
    .select('order_item_order_id', 'order_item_quantity', 'order_item_subtotal') \
    .groupBy('order_item_order_id')
    #.show() does not work since grouoped data has no attribute show

In [14]:
# help(order_items_grouped.sum())

In [16]:
order_items_grouped \
    .sum('order_item_quantity', 'order_item_subtotal') \
    .show()

+-------------------+------------------------+------------------------+
|order_item_order_id|sum(order_item_quantity)|sum(order_item_subtotal)|
+-------------------+------------------------+------------------------+
|                148|                       8|      479.99000549316406|
|                463|                      13|       829.9200096130371|
|                471|                       2|      169.98000717163086|
|                496|                       7|        441.950008392334|
|               1088|                       3|      249.97000885009766|
|               1580|                       5|      299.95001220703125|
|               1591|                      10|       439.8599967956543|
|               1645|                      14|       1509.790023803711|
|               2366|                       3|       299.9700012207031|
|               2659|                       8|       724.9100151062012|
|               2866|                       5|        569.960018

In [24]:
order_items_df = order_items_grouped \
    .sum('order_item_quantity', 'order_item_subtotal') \
    .toDF('order_item_order_id', 'total_quantity', 'total_revenue') 

In [26]:
type(order_items_df)

pyspark.sql.dataframe.DataFrame

In [27]:
order_items_df.show()

+-------------------+--------------+------------------+
|order_item_order_id|total_quantity|     total_revenue|
+-------------------+--------------+------------------+
|                148|             8|479.99000549316406|
|                463|            13| 829.9200096130371|
|                471|             2|169.98000717163086|
|                496|             7|  441.950008392334|
|               1088|             3|249.97000885009766|
|               1580|             5|299.95001220703125|
|               1591|            10| 439.8599967956543|
|               1645|            14| 1509.790023803711|
|               2366|             3| 299.9700012207031|
|               2659|             8| 724.9100151062012|
|               2866|             5|  569.960018157959|
|               3175|             4|209.97000122070312|
|               3749|             3|143.97000122070312|
|               3794|             5|299.95001220703125|
|               3918|             9| 829.9300155

In [28]:
from pyspark.sql.functions import round

In [31]:
# With this approach, only single aggregate funciton can be used
# To use multiple aggregate functions, use agg()

order_items_grouped. \
    sum('order_item_quantity', 'order_item_subtotal'). \
    toDF('order_item_order_id', 'total_quantity', 'total_revenue'). \
    withColumn('total_revenue', round('total_revenue', 2)). \
    show()

+-------------------+--------------+-------------+
|order_item_order_id|total_quantity|total_revenue|
+-------------------+--------------+-------------+
|                148|             8|       479.99|
|                463|            13|       829.92|
|                471|             2|       169.98|
|                496|             7|       441.95|
|               1088|             3|       249.97|
|               1580|             5|       299.95|
|               1591|            10|       439.86|
|               1645|            14|      1509.79|
|               2366|             3|       299.97|
|               2659|             8|       724.91|
|               2866|             5|       569.96|
|               3175|             4|       209.97|
|               3749|             3|       143.97|
|               3794|             5|       299.95|
|               3918|             9|       829.93|
|               3997|             4|       579.95|
|               4101|          